In [3]:
import mysql.connector
import pandas as pd
import os

# ==========================
# CONNECT MYSQL
# ==========================
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Diksha@2407",
    database="retaildb"
)

output_path = "powerbi_data"
os.makedirs(output_path, exist_ok=True)

# ==========================
# LOAD TABLES
# ==========================
sales = pd.read_sql("SELECT * FROM sales", conn)
sales_details = pd.read_sql("SELECT * FROM sales_details", conn)
product = pd.read_sql("SELECT * FROM product", conn)
category = pd.read_sql("SELECT * FROM category", conn)
customer = pd.read_sql("SELECT * FROM customer", conn)
inventory = pd.read_sql("SELECT * FROM inventory", conn)
supplier = pd.read_sql("SELECT * FROM supplier", conn)
purchase = pd.read_sql("SELECT * FROM purchase_pattern", conn)

# ==========================
# DATE CONVERSION
# ==========================
sales['sale_date'] = pd.to_datetime(sales['sale_date'])

sales['Day'] = sales['sale_date'].dt.day_name()
sales['Week'] = sales['sale_date'].dt.isocalendar().week
sales['Month'] = sales['sale_date'].dt.month_name()
sales['Year'] = sales['sale_date'].dt.year

# ==========================
# SALES FACT TABLE
# ==========================
fact_sales = sales.merge(sales_details, on='sale_id')
fact_sales = fact_sales.merge(product, on='product_id')
fact_sales = fact_sales.merge(category, on='category_id')
fact_sales = fact_sales.merge(customer, on='customer_id')

# ==========================
# KPI TABLES
# ==========================

# Daily Sales
daily_sales = fact_sales.groupby('sale_date').agg(
    Revenue=('total_price', 'sum'),
    Orders=('sale_id', 'nunique'),
    Quantity=('quantity', 'sum')
).reset_index()

# Weekly Sales
weekly_sales = fact_sales.groupby('Week').agg(
    Revenue=('total_price', 'sum'),
    Orders=('sale_id', 'nunique')
).reset_index()

# Monthly Sales
monthly_sales = fact_sales.groupby('Month').agg(
    Revenue=('total_price', 'sum'),
    Orders=('sale_id', 'nunique')
).reset_index()

# Yearly Sales
yearly_sales = fact_sales.groupby('Year').agg(
    Revenue=('total_price', 'sum')
).reset_index()

# Top Products
top_products = fact_sales.groupby('product_name').agg(
    Revenue=('total_price', 'sum'),
    Qty=('quantity', 'sum')
).reset_index().sort_values(by='Revenue', ascending=False)

# ==========================
# FIX LOW INVENTORY ERROR
# ==========================
inventory['stock_quantity'] = pd.to_numeric(inventory['stock_quantity'], errors='coerce')
inventory['reorder_level'] = pd.to_numeric(inventory['reorder_level'], errors='coerce')

# Fill missing reorder levels
inventory['reorder_level'] = inventory['reorder_level'].fillna(10)

# Fill missing stock values
inventory['stock_quantity'] = inventory['stock_quantity'].fillna(0)

# Low Stock Table
low_stock = inventory[
    inventory['stock_quantity'] < inventory['reorder_level']
]

# ==========================
# Frequent Customers
# ==========================
customers = fact_sales.groupby('name').agg(
    Orders=('sale_id', 'nunique'),
    Spend=('total_price', 'sum')
).reset_index().sort_values(by='Spend', ascending=False)

# ==========================
# EXPORT CSV FILES
# ==========================
daily_sales.to_csv(output_path + "/daily_sales.csv", index=False)
weekly_sales.to_csv(output_path + "/weekly_sales.csv", index=False)
monthly_sales.to_csv(output_path + "/monthly_sales.csv", index=False)
yearly_sales.to_csv(output_path + "/yearly_sales.csv", index=False)
top_products.to_csv(output_path + "/top_products.csv", index=False)
low_stock.to_csv(output_path + "/low_stock.csv", index=False)
customers.to_csv(output_path + "/customers.csv", index=False)

print("All Power BI files ready successfully.")

C:\Users\allab\AppData\Local\Temp\ipykernel_27072\2781745129.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sales = pd.read_sql("SELECT * FROM sales", conn)
C:\Users\allab\AppData\Local\Temp\ipykernel_27072\2781745129.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sales_details = pd.read_sql("SELECT * FROM sales_details", conn)
C:\Users\allab\AppData\Local\Temp\ipykernel_27072\2781745129.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  product = pd.read_sql("SELECT * FROM product", conn)
C:\Users\allab\AppDa

All Power BI files ready successfully.


In [4]:
import mysql.connector
import pandas as pd
import os

# ==========================
# CONNECT MYSQL
# ==========================
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Diksha@2407",
    database="retaildb"
)

output_path = "powerbi_data"
os.makedirs(output_path, exist_ok=True)

# ==========================
# LOAD TABLES
# ==========================
sales = pd.read_sql("SELECT * FROM sales", conn)
sales_details = pd.read_sql("SELECT * FROM sales_details", conn)
product = pd.read_sql("SELECT * FROM product", conn)
category = pd.read_sql("SELECT * FROM category", conn)
customer = pd.read_sql("SELECT * FROM customer", conn)

# ==========================
# DATE FORMAT
# ==========================
sales['sale_date'] = pd.to_datetime(sales['sale_date'])

# ==========================
# CREATE SALES DETAIL FACT TABLE
# ==========================
sales_detail_fact = sales.merge(sales_details, on='sale_id')
sales_detail_fact = sales_detail_fact.merge(product, on='product_id')
sales_detail_fact = sales_detail_fact.merge(category, on='category_id')
sales_detail_fact = sales_detail_fact.merge(customer, on='customer_id')

# ==========================
# KEEP ONLY REQUIRED COLUMNS
# ==========================
sales_detail_fact = sales_detail_fact[
    ['sale_date', 'product_name', 'category_name', 'name', 'quantity', 'total_price']
]

# ==========================
# RENAME COLUMNS
# ==========================
sales_detail_fact.columns = [
    'sale_date',
    'product_name',
    'category',
    'customer_name',
    'qty',
    'revenue'
]

# ==========================
# EXPORT CSV
# ==========================
sales_detail_fact.to_csv(output_path + "/sales_detail_fact.csv", index=False)

print("sales_detail_fact.csv created successfully.")

sales_detail_fact.csv created successfully.


C:\Users\allab\AppData\Local\Temp\ipykernel_27072\4051286404.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sales = pd.read_sql("SELECT * FROM sales", conn)
C:\Users\allab\AppData\Local\Temp\ipykernel_27072\4051286404.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sales_details = pd.read_sql("SELECT * FROM sales_details", conn)
C:\Users\allab\AppData\Local\Temp\ipykernel_27072\4051286404.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  product = pd.read_sql("SELECT * FROM product", conn)
C:\Users\allab\AppDa